In [0]:
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
from pyspark.ml import Pipeline

df = spark.table("workspace.default.gold_hospital_features")

# Select features
feature_cols = [
    "age_numeric", "time_in_hospital", "num_lab_procedures",
    "num_procedures", "num_medications", "number_outpatient",
    "number_emergency", "number_inpatient", "admission_type_id",
    "discharge_disposition_id", "admission_source_id",
    "high_medication_flag", "high_procedures_flag",
    "frequent_visitor_flag", "long_stay_flag",
    "insulin_flag", "diabetes_primary_flag"
]

df_ml = df.select(feature_cols + ["readmitted_binary"]).na.drop()

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_ml = assembler.transform(df_ml).select("features", "readmitted_binary")

train_df, test_df = df_ml.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()} | Test: {test_df.count()}")

Train: 77585 | Test: 19523


In [0]:
import mlflow
import mlflow.spark
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

mlflow.set_experiment("/Hospital_Readmission_Prediction")

# Point MLflow to your UC volume
mlflow_path = "/Volumes/workspace/default/hospital_operations_ai"

with mlflow.start_run(run_name="RandomForest_v1"):

    rf = RandomForestClassifier(
        labelCol="readmitted_binary",
        featuresCol="features",
        numTrees=100,
        maxDepth=6,
        seed=42
    )

    model = rf.fit(train_df)
    predictions = model.transform(test_df)

    auc_eval = BinaryClassificationEvaluator(
        labelCol="readmitted_binary", metricName="areaUnderROC")
    acc_eval = MulticlassClassificationEvaluator(
        labelCol="readmitted_binary", metricName="accuracy")
    f1_eval = MulticlassClassificationEvaluator(
        labelCol="readmitted_binary", metricName="f1")

    auc = auc_eval.evaluate(predictions)
    acc = acc_eval.evaluate(predictions)
    f1  = f1_eval.evaluate(predictions)

    mlflow.log_param("numTrees", 100)
    mlflow.log_param("maxDepth", 6)
    mlflow.log_metric("AUC", auc)
    mlflow.log_metric("Accuracy", acc)
    mlflow.log_metric("F1_Score", f1)
    mlflow.spark.log_model(
        model, 
        "random_forest_model",
        dfs_tmpdir=mlflow_path
    )

    print(f"AUC:      {auc:.4f}")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Score: {f1:.4f}")

2026/03/10 11:07:58 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 11:08:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-5bab0994-4849-441a-8053-55/tmpms53_577/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/10 11:08:03 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


AUC:      0.6699
Accuracy: 0.6248
F1 Score: 0.6193


In [0]:
from pyspark.sql.functions import col

predictions.select(
    "features",
    col("readmitted_binary").alias("actual"),
    col("prediction").alias("predicted"),
    col("probability")
).write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_predictions")

print("Predictions saved to Gold layer")

Predictions saved to Gold layer


In [0]:
import pandas as pd

feature_importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.featureImportances.toArray()
}).sort_values("importance", ascending=False)

print(feature_importance.to_string(index=False))

                 feature  importance
        number_inpatient    0.514289
   frequent_visitor_flag    0.179633
        number_emergency    0.082786
       number_outpatient    0.052492
     admission_source_id    0.046586
         num_medications    0.033458
             age_numeric    0.028739
      num_lab_procedures    0.012007
        time_in_hospital    0.011573
discharge_disposition_id    0.011302
    high_medication_flag    0.007246
       admission_type_id    0.007061
          num_procedures    0.006842
   diabetes_primary_flag    0.002468
            insulin_flag    0.002152
          long_stay_flag    0.000839
    high_procedures_flag    0.000525


In [0]:
with mlflow.start_run(run_name="RandomForest_v2_tuned"):

    rf2 = RandomForestClassifier(
        labelCol="readmitted_binary",
        featuresCol="features",
        numTrees=200,
        maxDepth=8,
        minInstancesPerNode=5,
        seed=42
    )

    model2 = rf2.fit(train_df)
    predictions2 = model2.transform(test_df)

    auc2 = auc_eval.evaluate(predictions2)
    acc2 = acc_eval.evaluate(predictions2)
    f12  = f1_eval.evaluate(predictions2)

    mlflow.log_param("numTrees", 200)
    mlflow.log_param("maxDepth", 8)
    mlflow.log_param("minInstancesPerNode", 5)
    mlflow.log_metric("AUC", auc2)
    mlflow.log_metric("Accuracy", acc2)
    mlflow.log_metric("F1_Score", f12)
    mlflow.spark.log_model(
        model2,
        "random_forest_model_v2",
        dfs_tmpdir="/Volumes/workspace/default/hospital_operations_ai"
    )

    print(f"V2 AUC:      {auc2:.4f}")
    print(f"V2 Accuracy: {acc2:.4f}")
    print(f"V2 F1 Score: {f12:.4f}")

2026/03/10 11:19:17 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/03/10 11:19:19 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-5bab0994-4849-441a-8053-55/tmpit0q1ro9/model, flavor: spark). Fall back to return ['pyspark==4.0.0']. Set logging level to DEBUG to see the full traceback. 
2026/03/10 11:19:19 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


V2 AUC:      0.6742
V2 Accuracy: 0.6287
V2 F1 Score: 0.6225
